# Model Comparison Across HistGradientBoosting, XGBoost, and Balanced Random Forest

This notebook reads the standardized final-row CSV exports from the three model notebooks and ranks them using the imbalance-aware rule agreed for the project.

Run the final evaluation cell in each model notebook first so these files exist:
- `histgradient_final_row.csv`
- `xgboost_final_row.csv`
- `balanced_rf_final_row.csv`


In [7]:
from pathlib import Path

import pandas as pd

files = {
    "HistGradientBoosting": Path("data/processed/histgradient_final_row.csv"),
    "XGBoost": Path("data/processed/xgboost_final_row.csv"),
    "Balanced RF (tuned)": Path("data/processed/balanced_rf_final_row.csv"),
}

missing = [name for name, path in files.items() if not path.exists()]
if missing:
    missing_text = ", ".join(missing)
    raise FileNotFoundError(
        "Missing comparison inputs for: " + missing_text + "\n"
        + "Run the final evaluation cell in each corresponding notebook first so it exports its *_final_row.csv file."
    )

comparison_table = pd.concat([pd.read_csv(path) for path in files.values()], ignore_index=True)

ordered_columns = [
    "Model",
    "learning_rate",
    "max_depth",
    "max_iter",
    "min_samples_leaf",
    "n_estimators",
    "max_features",
    "Selection_Objective",
    "Selection_Score",
    "Accuracy",
    "Balanced_Accuracy",
    "Precision_0",
    "Recall_0",
    "F1_0",
    "PR_AUC_0",
    "Precision_1",
    "Recall_1",
    "F1_1",
    "PR_AUC_1",
    "ROC_AUC_1",
]

comparison_table = comparison_table[ordered_columns]
ranked_models = comparison_table.sort_values(
    by=["Balanced_Accuracy", "Recall_0", "F1_0", "PR_AUC_0", "ROC_AUC_1", "F1_1"],
    ascending=False,
).reset_index(drop=True)

print("Ranking rule: Balanced_Accuracy > Recall_0 > F1_0 > PR_AUC_0 > ROC_AUC_1 > F1_1")
ranked_models.to_csv("data/processed/model_comparison_ranked.csv", index=False)
ranked_models


Ranking rule: Balanced_Accuracy > Recall_0 > F1_0 > PR_AUC_0 > ROC_AUC_1 > F1_1


,Model,learning_rate,max_depth,max_iter,min_samples_leaf,n_estimators,max_features,Selection_Objective,Selection_Score,Accuracy,Balanced_Accuracy,Precision_0,Recall_0,F1_0,PR_AUC_0,Precision_1,Recall_1,F1_1,PR_AUC_1,ROC_AUC_1
0,Balanced RF (tuned),NaN,10,NaN,10.0,700.0,sqrt,Best_CV_PR_AUC_0,0.1586,0.7970,0.6043,0.1608,0.3758,0.2252,0.1597,0.9400,0.8329,0.8832,0.9448,0.6378
1,XGBoost,0.1,5,NaN,NaN,200.0,NaN,Best_CV_F1,0.9347,0.8725,0.5579,0.1859,0.1847,0.1853,0.1435,0.9306,0.9311,0.9308,0.9408,0.6157
2,HistGradientBoosting,0.1,7,200.0,20.0,NaN,NaN,Best_CV_F1,0.9508,0.9050,0.5406,0.2537,0.1083,0.1518,0.1415,0.9276,0.9729,0.9497,0.9448,0.6271
